# Train Layer 2 (mDeBERTa) on dataset v3 (public data) on Colab
Runtime > Change runtime type > **T4 GPU**. Run the cells in order (about 25-40 minutes, the dataset is larger than v2).
Results are written **straight to Google Drive**, so a disconnected runtime cannot lose them.
Before starting, on your PC run `python models/l2_multilingual/make_colab_bundle.py --public` and have `l2_colab_bundle_public.zip` ready.
The old model in `models/l2_multilingual/artifacts` is NOT touched: the new one goes to `ai_firewall/l2_artifacts_public`.

In [ ]:
!pip -q install transformers sentencepiece accelerate pandas scikit-learn
!nvidia-smi | head -15

In [ ]:
# 1. Mount Drive (a pop-up asks for permission)
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ai_firewall/l2_artifacts_public'
!mkdir -p {OUT}

In [ ]:
# 2. Upload l2_colab_bundle_public.zip when asked
from google.colab import files
up = files.upload()
!unzip -o l2_colab_bundle_public.zip -d /content/proj
%cd /content/proj

In [ ]:
# 3. Quick check on 40 documents (1-2 minutes). Stop here if it errors. Not saved to Drive.
# %cd is repeated here: a runtime restart or a skipped cell 2 resets the working directory to /content.
%cd /content/proj
!ls models/l2_multilingual/train.py
!python models/l2_multilingual/train.py --data-dir data/processed_public --limit 40 --epochs 1 --out /content/smoke

In [ ]:
# 4. Real training (same settings as the first model: 3 epochs, batch 16, lr 2e-5, fp16). Output goes directly to Drive.
%cd /content/proj
!python models/l2_multilingual/train.py --data-dir data/processed_public --epochs 3 --out {OUT}

In [ ]:
# 5. Check the files are in Drive. model.safetensors must be about 1.1-1.3 GB.
!ls -la {OUT} {OUT}/checkpoint

## Bring back to the project
On your PC, open Google Drive > `ai_firewall/l2_artifacts_public`, download the folder, and place it in `models/l2_multilingual/` named `artifacts_public` (keep `artifacts` as it is, it is the old model used for the comparison). It contains:
- `checkpoint/` (model + tokenizer, ~1.1 GB, **git-ignored**)
- `temperature.json`, `metrics.json`, `train_log.json`, `test_scores.csv` (small)
Then tell Claude: the evaluation compares the old and the new model on the same held-out sets.